# Matrix Operations

| Difficulty | ██░░░░░░░░ 2/10 |
| :--- | :--- |
| Prerequisites | 01.01 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/02_matrix_operations.ipynb)

---

## 🎯 Learning Objective

Learn the core matrix operations — multiplication, transpose, and rank — and the four fundamental subspaces associated with a matrix.

---

## 📐 Theory

### Matrix multiplication

Given $A$ of shape $(m,n)$ and $B$ of shape $(n,p)$, their product $C = AB$ has shape $(m,p)$
with entries

$$C_{ik} = \sum_{j=1}^{n} A_{ij}B_{jk}$$

Entry $(i,k)$ of the product is the dot product of row $i$ of $A$ with column $k$ of $B$ —
matrix multiplication is nothing but a grid of dot products.

### Transpose

The transpose $A^\top$ flips rows and columns: $(A^\top)_{ij} = A_{ji}$. A useful identity
that comes up constantly when deriving gradients: $(AB)^\top = B^\top A^\top$ (the order
reverses).

### Rank and the four fundamental subspaces

The **column space** of $A$ is the span of its columns — every possible output $A\mathbf{x}$.
The **row space** is the span of its rows. The **rank** of $A$ is the dimension of the column
space, which always equals the dimension of the row space (a genuinely non-obvious fact). The
**null space** is $\{\mathbf{x} \mid A\mathbf{x} = \mathbf{0}\}$ — every input that gets
crushed to zero.

These are tied together by the **rank-nullity theorem**: for an $A$ with $n$ columns,

$$\text{rank}(A) + \dim(\text{null}(A)) = n$$

Intuitively: every input dimension either survives into the output (contributing to rank) or
gets annihilated (contributing to the null space) — there's nowhere else for it to go.

---

## 👁️ Visual Intuition

Applying a matrix to a whole grid of points shows rank directly: a full-rank $2\times 2$
matrix keeps the grid two-dimensional, while a rank-deficient matrix squashes it onto a line.

In [ ]:
# Rank made visible: a full-rank matrix keeps the grid 2D, a rank-deficient one flattens it
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

xs, ys = np.meshgrid(np.linspace(-1, 1, 15), np.linspace(-1, 1, 15))
grid = np.stack([xs.ravel(), ys.ravel()])   # shape (2, 225)

A_full = np.array([[2.0, 1.0], [0.0, 1.5]])          # full rank: columns point in different directions
A_deficient = np.array([[1.0, 2.0], [0.5, 1.0]])      # row 2 = 0.5 * row 1 -> rank 1

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, A, title in zip(axes, [A_full, A_deficient],
                         ["Full rank: stays 2D", "Rank-deficient: collapses to a line"]):
    transformed = A @ grid
    ax.scatter(transformed[0], transformed[1], s=8, color="#4C72B0")
    ax.set_title(f"{title}\nrank = {np.linalg.matrix_rank(A)}")
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Matrix multiplication from three nested loops makes the summation formula concrete. Then we
verify the rank-nullity theorem computationally on a matrix with genuine rank deficiency.

In [ ]:
# Matrix multiplication from scratch, plus a rank-nullity theorem sanity check
import numpy as np
from scipy.linalg import null_space

def matmul_from_scratch(A, B):
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, "inner dimensions must match"
    C = np.zeros((m, p))
    for i in range(m):
        for k in range(p):
            total = 0.0
            for j in range(n):
                total += A[i, j] * B[j, k]
            C[i, k] = total
    return C

A = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
B = np.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]])
print("matmul_from_scratch matches A @ B:", np.allclose(matmul_from_scratch(A, B), A @ B))
print("(AB)^T == B^T A^T:                ", np.allclose((A @ B).T, B.T @ A.T))

M = np.array([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [1.0, 0.0, 1.0]])   # row 2 = 2 * row 1 -> rank-deficient
rank = np.linalg.matrix_rank(M)
ns = null_space(M)
print(f"\nrank(M) = {rank},  dim(null(M)) = {ns.shape[1]},  n (columns) = {M.shape[1]}")
print("Rank-nullity theorem holds:", rank + ns.shape[1] == M.shape[1])

## 🤖 Why This Matters for AI

Every dense layer in a neural network computes $\mathbf{y} = W\mathbf{x} + \mathbf{b}$ — the
weight matrix $W$ **is** a learned linear transformation. The rank of $W$ caps how much of the
input's information can possibly survive: a rank-$k$ weight matrix can only produce outputs
that are combinations of $k$ fixed directions, no matter what the input is. This is precisely
the bottleneck that low-rank adaptation methods like LoRA exploit deliberately — a preview of
[Notebook 01.07](07_SVD_and_PCA.ipynb).

In [ ]:
# A rank-2 weight matrix caps EVERY possible output to a 2D subspace, regardless of input
import numpy as np

rng = np.random.default_rng(0)
d_in, d_out = 6, 6

W_full = rng.standard_normal((d_out, d_in))
print("Full random weight matrix rank:", np.linalg.matrix_rank(W_full), "out of", min(d_in, d_out))

# Force a rank-2 weight matrix, built as a sum of two rank-1 outer products
u1, u2 = rng.standard_normal(d_out), rng.standard_normal(d_out)
v1, v2 = rng.standard_normal(d_in), rng.standard_normal(d_in)
W_low_rank = np.outer(u1, v1) + np.outer(u2, v2)
print("Constrained weight matrix rank:", np.linalg.matrix_rank(W_low_rank))

xs = rng.standard_normal((10, d_in))          # 10 different random inputs
outputs = xs @ W_low_rank.T                    # every one of the 10 outputs...
recovered_rank = np.linalg.matrix_rank(outputs)
print("\nRank of 10 stacked outputs from W_low_rank:", recovered_rank)
print("No matter which/how many inputs you try, outputs never escape a", recovered_rank, "-dimensional subspace.")

## 🏋️ Exercises

### Warm-up
1. Multiply $\begin{pmatrix}1&2\\3&4\end{pmatrix}$ by $\begin{pmatrix}5&6\\7&8\end{pmatrix}$ by hand using the summation formula, then verify with `matmul_from_scratch`.

### Practice
2. For the matrix $M = \begin{pmatrix}1&2&1\\2&4&2\\1&1&2\end{pmatrix}$, compute its rank and null space with `np.linalg.matrix_rank`/`scipy.linalg.null_space`, and verify the rank-nullity theorem holds.

### Challenge
3. Construct a $5\times5$ matrix with rank exactly 3 by summing three rank-1 outer products (like `W_low_rank` above), verify its rank, then compute its null space and confirm the null space dimension is exactly $5 - 3 = 2$.

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 3 (The Four Fundamental Subspaces)
- [3Blue1Brown: Essence of Linear Algebra](https://www.3blue1brown.com/topics/linear-algebra) (video series)

---

*Next notebook: [Linear Transformations](03_linear_transformations.ipynb)*